##1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


##2. Install Required packages

In [ ]:
!pip -q install pydicom tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.1 MB/s eta 0:00:00


##3. Import necessary libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import pydicom

from tqdm import tqdm

import os

##3. Define Dataset paths

In [ ]:
DATASET_PATH = Path(
    "/content/drive/MyDrive/Dissertation/Dataset"
)


RAW_PATH = DATASET_PATH / "Raw"

PROCESSED_PATH = DATASET_PATH / "Processed"


CONSENSUS_FILE = (
    PROCESSED_PATH /
    "consensus_nodule_metadata_v2.csv"
)


consensus_df = pd.read_csv(
    CONSENSUS_FILE
)


print(
    consensus_df.shape
)

(6407, 9)


##5. Check Available Processed CT Volumes

In [ ]:
hu_files = sorted(
    PROCESSED_PATH.glob("*_HU.npy")
)


print(
    "Processed volumes:",
    len(hu_files)
)

print(
    hu_files[:5]
)

Processed volumes: 1
[PosixPath('/content/drive/MyDrive/Dissertation/Dataset/Processed/LIDC-IDRI-0001_HU.npy')]


##6. Check Matching Patients

In [ ]:
consensus_df["PatientID"] = (
    consensus_df["StudyUID"]
)


print(
    consensus_df.head()
)

      ConsensusID  XMLFile                                           StudyUID  \
0       000.xml_0  000.xml  1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...   
1    000.xml_0102  000.xml  1.3.6.1.4.1.14519.5.2.1.6279.6001.261049959740...   
2       000.xml_1  000.xml  1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...   
3  000.xml_124130  000.xml  1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...   
4  000.xml_124241  000.xml  1.3.6.1.4.1.14519.5.2.1.6279.6001.213637791937...   

  NoduleID  Readers  MeanMalignancy  MedianMalignancy  ROICount  CancerLabel  \
0        0        1             1.0               1.0        13            0   
1     0102        1             3.0               3.0         4            0   
2        1        2             2.5               2.5        11            0   
3   124130        1             1.0               1.0        12            0   
4   124241        1             1.0               1.0         7            0   

                                

##7. Build StudyUID → Patient Folder Mapping

###7.1. Find All Raw Patients

In [ ]:
raw_patients = sorted(
    [
        p for p in RAW_PATH.iterdir()
        if p.is_dir()
    ]
)


print(
    "Raw patients:",
    len(raw_patients)
)


print(raw_patients[:5])

Raw patients: 10
[PosixPath('/content/drive/MyDrive/Dissertation/Dataset/Raw/LIDC-IDRI-0001'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/Raw/LIDC-IDRI-0002'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/Raw/LIDC-IDRI-0003'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/Raw/LIDC-IDRI-0004'), PosixPath('/content/drive/MyDrive/Dissertation/Dataset/Raw/LIDC-IDRI-0005')]


###7.2. Function to Extract Study UID

In [ ]:
def get_patient_study_uid(patient_folder):

    for dcm_file in patient_folder.rglob("*.dcm"):

        try:

            ds = pydicom.dcmread(
                dcm_file,
                stop_before_pixels=True
            )

            return ds.StudyInstanceUID

        except Exception:
            continue


    return None

###7.3. Create Mapping Table

In [ ]:
study_mapping = []


for patient_folder in tqdm(
    raw_patients,
    desc="Reading DICOM folders"
):

    study_uid = get_patient_study_uid(
        patient_folder
    )


    study_mapping.append(
        {
            "PatientFolder": patient_folder.name,
            "StudyUID": study_uid
        }
    )


study_mapping_df = pd.DataFrame(
    study_mapping
)


study_mapping_df

Reading DICOM folders: 100%|██████████| 10/10 [00:00<00:00, 88.53it/s]


,PatientFolder,StudyUID
0,LIDC-IDRI-0001,1.3.6.1.4.1.14519.5.2.1.6279.6001.298806137288...
1,LIDC-IDRI-0002,1.3.6.1.4.1.14519.5.2.1.6279.6001.490157381160...
2,LIDC-IDRI-0003,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...
3,LIDC-IDRI-0004,1.3.6.1.4.1.14519.5.2.1.6279.6001.191425307197...
4,LIDC-IDRI-0005,1.3.6.1.4.1.14519.5.2.1.6279.6001.190188259083...
5,LIDC-IDRI-0006,1.3.6.1.4.1.14519.5.2.1.6279.6001.324680252006...
6,LIDC-IDRI-0007,1.3.6.1.4.1.14519.5.2.1.6279.6001.280315210397...
7,LIDC-IDRI-0008,1.3.6.1.4.1.14519.5.2.1.6279.6001.185810436275...
8,LIDC-IDRI-0009,1.3.6.1.4.1.14519.5.2.1.6279.6001.225213110794...
9,LIDC-IDRI-0010,1.3.6.1.4.1.14519.5.2.1.6279.6001.303099231937...


###7.4. Check Annotation Matching

In [ ]:
matched = consensus_df.merge(
    study_mapping_df,
    on="StudyUID",
    how="inner"
)


print(
    "Total nodules:",
    len(consensus_df)
)

print(
    "Matched nodules:",
    len(matched)
)

Total nodules: 6407
Matched nodules: 61


###7.5. Save the matched metadata

In [ ]:
matched_path = (
    PROCESSED_PATH /
    "matched_consensus_metadata_v2.csv"
)


matched.to_csv(
    matched_path,
    index=False
)


print(
    "Saved:",
    matched_path
)

Saved: /content/drive/MyDrive/Dissertation/Dataset/Processed/matched_consensus_metadata_v2.csv


##8. Extract ROI Coordinates from XML
####We need to locate the XML file corresponding to each matched nodule.

The next cells will:

1. Read the XML folder.
2. Find the matching XML file.
3. Extract the ROI information.
4. Attach it to the matched metadata.

###8.1. Define XML Path

In [ ]:
XML_ROOT = Path(
    "/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING"
)


xml_files = list(
    XML_ROOT.rglob("*.xml")
)


print(
    "XML files:",
    len(xml_files)
)

XML files: 1319


###8.2. Create XML Lookup

In [ ]:
xml_lookup = {
    xml.name: xml
    for xml in xml_files
}


print(
    list(xml_lookup.items())[:3]
)

[('161-resubmitted-correction-3-9-12.xml', PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/161-resubmitted-correction-3-9-12.xml')), ('001.xml', PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/186/001.xml')), ('006.xml', PosixPath('/content/drive/MyDrive/Dissertation/Dataset/ANNOTATION_STAGING/LIDC-XML-only/tcia-lidc-xml/186/006.xml'))]


###8.3. Load Matched Metadata

In [ ]:
matched_df = pd.read_csv(
    PROCESSED_PATH /
    "matched_consensus_metadata.csv"
)


print(
    matched_df.shape
)


matched_df.head()

(61, 11)


,ConsensusID,StudyUID,NoduleID,Readers,MeanMalignancy,MedianMalignancy,ROICount,XMLFile,CancerLabel,PatientID,PatientFolder
0,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,0,1,4.0,4.0,10,072.xml,1,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,LIDC-IDRI-0003
1,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,12,1,2.0,2.0,8,072.xml,0,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,LIDC-IDRI-0003
2,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,8,1,2.0,2.0,4,072.xml,0,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,LIDC-IDRI-0003
3,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,9,1,2.0,2.0,8,072.xml,0,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,LIDC-IDRI-0003
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,IL057_130591,1,5.0,5.0,7,072.xml,1,1.3.6.1.4.1.14519.5.2.1.6279.6001.101370605276...,LIDC-IDRI-0003


##9. Import XML Parser

In [ ]:
import xml.etree.ElementTree as ET

##10. Define XML Namespace

In [ ]:
def get_namespace(root):

    namespace = root.tag.split("}")[0].replace("{","")

    return {
        "nih": namespace
    }


NS = get_namespace(root)

print(NS)

{'nih': 'http://www.nih.gov/idri'}


##11. Function to Load XML

In [ ]:
def load_xml(xml_path):

    tree = ET.parse(xml_path)

    return tree.getroot()